In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from shapely.geometry import mapping
import rasterio
from rasterio.enums import Resampling

In [ ]:
# 获取用户上传的分类权重 CSV 文件，并创建一个字典，将 flag_value 和 Onshore wind 权重建立映射
lucc = xr.open_dataset(r"F:\Workspace\DATA_Landuse\World_landuse_025d.tif")["band_data"][0, :, :]
classification_df = pd.read_csv("land_use_classification.csv")
lucc_weights = dict(zip(classification_df['Flag Value'], classification_df['Onshore wind-Base']))
lucc_cf = dict(zip(classification_df['Flag Value'], classification_df['CF']))
# 定义函数，将土地利用类型的值映射到对应的权重
def lucc_process_weighted(x):
    return lucc_weights.get(x, np.nan)  # 如果找不到对应的值，则返回 NaN

def lucc_cf_weighted(x):
    return lucc_cf.get(x, np.nan)  # 如果找不到对应的值，则返回 NaN
# 使用 map_blocks 将土地利用类型数据按块映射为对应权重

lucc_cf = lucc.map_blocks(np.vectorize(lucc_cf_weighted))
lucc = lucc.map_blocks(np.vectorize(lucc_process_weighted))

In [ ]:
CF_wind_list = []
CF_solar_list = []
for i in range(0, 1440, 60):
    print(i)
    lucc_cf_chunk = lucc_cf[:, i:i+60]
    data = xr.open_dataset(rf"D:\Workspace\average_2004_2023_segment_{i}.nc", engine='netcdf4')
    CF_wind = np.mean(data["CF_wind_avg"][0:8760, :, :], axis=0) * lucc_cf_chunk
    CF_wind_list.append(CF_wind)
    
    CF_solar = np.mean(data["CF_pv_avg"][0:8760, :, :], axis=0) * lucc_cf_chunk
    CF_solar_list.append(CF_solar)    

CF_wind_all = xr.concat(CF_wind_list, dim="longitude")
CF_solar_all = xr.concat(CF_solar_list, dim="longitude")

In [ ]:
import matplotlib.pyplot as plt

# 假设 CF_wind_all 是你的数据
plt.imshow(CF_wind_all)
plt.colorbar()

# 保存图像
plt.savefig('CF_wind_avg.png',dpi=600)  # 你可以更改文件名和格式，如 'output_image.jpg' 或 'output_image.pdf'

# 如果你想在保存后显示图像，可以调用 plt.show()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 假设 CF_wind_all 是你的数据
plt.imshow(CF_solar_all)
plt.colorbar()

# 保存图像
plt.savefig('CF_solar_avg.png',dpi=600)  # 你可以更改文件名和格式，如 'output_image.jpg' 或 'output_image.pdf'

# 如果你想在保存后显示图像，可以调用 plt.show()
plt.show()